In [ ]:
import zipfile
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
import joblib
from sklearn.metrics import f1_score, classification_report
import os
from google.colab import drive
from sklearn.base import clone
import time

In [ ]:
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/thesis_results/AAPD_SVC_final"
os.makedirs(output_dir, exist_ok=True)

Mounted at /content/drive


# Data inspection

In [ ]:
with zipfile.ZipFile("aapd.zip") as z:
    with z.open("aapd.json") as f:
        aapd = json.load(f)

print(type(aapd))
print((aapd.keys()))
print(aapd["data"].keys())

<class 'dict'>
dict_keys(['meta', 'label_set', 'data'])
dict_keys(['train', 'val', 'test'])


In [ ]:
aapd["meta"]

{'name': 'AAPD',
 'source': 'https://link.springer.com/article/10.1007/s10579-022-09623-2',
 'n_train': 53840,
 'n_val': 1000,
 'n_test': 1000,
 'n_labels': 54,
 'avg_n_lbls_per_text': 2.41,
 'min_lbl': 350,
 'max_lbl': 17152,
 'min_lbls_per_text': 2,
 'max_lbls_per_text': 8}

In [ ]:
aapd_label_set = aapd["label_set"]
aapd_label_set

['Adaptation and Self-Organizing Systems',
 'Applications',
 'Artificial Intelligence',
 'Combinatorics',
 'Computation and Language',
 'Computational Complexity',
 'Computational Engineering, Finance, and Science',
 'Computational Geometry',
 'Computational Linguistics',
 'Computer Science and Game Theory',
 'Computer Vision and Pattern Recognition',
 'Computers and Society',
 'Cryptography and Security',
 'Data Analysis, Statistics and Probability',
 'Data Structures and Algorithms',
 'Databases',
 'Digital Libraries',
 'Discrete Mathematics',
 'Disordered Systems and Neural Networks',
 'Distributed, Parallel, and Cluster Computing',
 'Formal Languages and Automata Theory',
 'Human-Computer Interaction',
 'Information Retrieval',
 'Information Theory (Computer Science)',
 'Information Theory (Mathematics)',
 'Logic',
 'Logic in Computer Science',
 'Machine Learning (Computer Science)',
 'Machine Learning (Statistics)',
 'Mathematical Software',
 'Methodology',
 'Multiagent Systems',


In [ ]:
aapd_df_train = pd.DataFrame(aapd["data"]["train"])
aapd_df_val = pd.DataFrame(aapd["data"]["val"])
aapd_df_test = pd.DataFrame(aapd["data"]["test"])

In [ ]:
aapd_df_train

,id,text,labels
0,train0,the relation between pearson 's correlation co...,"[Information Retrieval, Methodology]"
1,train1,the present work studies quantum and classical...,"[Quantum Physics, Information Theory (Computer..."
2,train2,one of the most important tasks in image proce...,"[Applications, Computer Vision and Pattern Rec..."
3,train3,frequency diverse \( fd \) radar waveforms are...,"[Information Theory (Computer Science), Inform..."
4,train4,unsupervised word embeddings have been shown t...,"[Computation and Language, Artificial Intellig..."
...,...,...,...
53835,train53835,this volume contains papers presented at wlpe ...,"[Programming Languages, Logic in Computer Scie..."
53836,train53836,in this paper we analyse belief propagation ov...,"[Artificial Intelligence, Statistical Mechanic..."
53837,train53837,inference problems in graphical models are oft...,"[Machine Learning (Computer Science), Machine ..."
53838,train53838,the classical emd algorithm has been used exte...,"[Numerical Analysis (Mathematics), Numerical A..."


In [ ]:
#checking if there's no overlap between the data in the splits
train_texts_set = set(aapd_df_train["text"])
val_texts_set = set(aapd_df_val["text"])
test_texts_set = set(aapd_df_test["text"])

train_val_overlap = train_texts_set & val_texts_set
train_test_overlap = train_texts_set & test_texts_set

print(f"Train-val overlap: {len(train_val_overlap)} examples")
print(f"Train-test overlap: {len(train_test_overlap)} examples")
#there is a minor overlap - I will not fix this since that would make my results incomparable to other existing AAPD baselines

Train-val overlap: 4 examples
Train-test overlap: 5 examples


In [ ]:
label_list = sorted(aapd_label_set)
mlb = MultiLabelBinarizer(classes=label_list)
mlb.fit(aapd_df_train["labels"])
joblib.dump(mlb, f"{output_dir}/mlb.joblib", compress=3)

['/content/drive/MyDrive/thesis_results/AAPD_SVC/mlb.joblib']

In [ ]:
aapd_y_train = mlb.transform(aapd_df_train["labels"])
aapd_y_val   = mlb.transform(aapd_df_val["labels"])
aapd_y_test  = mlb.transform(aapd_df_test["labels"])

print(aapd_y_train[:1]) #looks good

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]


In [ ]:
aapd_y_train.shape, aapd_y_val.shape, aapd_y_test.shape #ok

((53840, 54), (1000, 54), (1000, 54))

In [ ]:
label_counts = aapd_y_train.sum(axis=0)
min_count = label_counts.min()
max_count = label_counts.max()

print(min_count) #same as in the provided dataset statistics
print(max_count)

350
17152


In [ ]:
print(label_counts)
print(f"Average number of samples per label: {label_counts.mean()}")

[  350   628  4991  3355  2831  3077   923  1080   856  1720  2774  1572
  2722   385  4673  1267   867  4074   440  2258   915  1005  2018 17152
 17152   665  3510  8007  4939   484   503   956   711  3081  2024   408
   386  1052   945  3043   687  3329  1194  1381   376  1581   921  4003
  1261   664   875   875   390  2242]
Average number of samples per label: 2399.5925925925926


In [ ]:
aapd_X_train = aapd_df_train["text"]
aapd_X_val   = aapd_df_val["text"]
aapd_X_test  = aapd_df_test["text"]

aapd_X_train.shape, aapd_X_val.shape, aapd_X_test.shape

((53840,), (1000,), (1000,))

In [ ]:
#checking the length of abstracts in the dataset
df_len = pd.concat([
    pd.DataFrame({"abstract": aapd_X_train, "split": "train"}),
    pd.DataFrame({"abstract": aapd_X_val, "split": "val"}),
    pd.DataFrame({"abstract": aapd_X_test, "split": "test"})], ignore_index=True)

df_len["word_len"] = df_len["abstract"].str.split().str.len()
df_len[["word_len"]].describe()

,word_len
count,55840.000000
mean,163.428940
std,67.600259
min,1.000000
25%,114.000000
50%,157.000000
75%,208.000000
max,522.000000


In [ ]:
#checking the one with len == 1
df_len[df_len["word_len"] == 1][["split", "word_len", "abstract"]]
#will keep it anyway so as not to make changes to the dataset/keep the original splits

,split,word_len,abstract
25711,train,1,withdrawn\n
33668,train,1,withdrawn\n


# SVC baseline

In [ ]:
import os
print("CPU cores:", os.cpu_count())

CPU cores: 2


In [ ]:
svc_model = Pipeline([("tfidf", TfidfVectorizer(analyzer="word", lowercase=True)),
                    ("clf", OneVsRestClassifier(LinearSVC(dual="auto", max_iter=2000, random_state=42)))])

params = {"tfidf__max_features": [30000, 50000], #for BESS - try lower
          "tfidf__ngram_range": [(1,1), (1,2)],
          "tfidf__min_df": [2, 5],
          "tfidf__max_df": [0.95],   #removes most common words (removed 1 to make gs smaller)
          "clf__estimator__class_weight": ["balanced"], #(removed False to make gs smaller)
          "clf__estimator__C": [0.1, 1, 10]}

gs = GridSearchCV(
    svc_model,
    params,
    cv=3,
    n_jobs=2,
    verbose=2,
    refit=True,
    scoring="f1_macro") #in the paper introducing the AAPD, f1 micro was used for hyperparameter selection, but for my needs f1-macro i more suitable


start_train = time.perf_counter()

gs.fit(aapd_X_train, aapd_y_train)

end_train = time.perf_counter()
train_time = end_train - start_train
print(f"Training time: {train_time:.2f} seconds.")

joblib.dump(gs.best_estimator_, f"{output_dir}/AAPD_SVC_gs_best_model.joblib", compress=5)


print(gs.best_params_)
print(gs.best_score_)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
Training time: 18707.12 seconds.
{'clf__estimator__C': 1, 'clf__estimator__class_weight': 'balanced', 'tfidf__max_df': 0.95, 'tfidf__max_features': 50000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2)}
0.5642913051072358


In [ ]:
gs.best_estimator_

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, max_features=50000, min_df=5,
                                 ngram_range=(1, 2))),
                ('clf',
                 OneVsRestClassifier(estimator=LinearSVC(C=1,
                                                         class_weight='balanced',
                                                         max_iter=2000,
                                                         random_state=42)))])

In [ ]:
#unfitted copy of the best pipeline found by grid search
final_model = clone(gs.best_estimator_)
final_train_start = time.perf_counter()
final_model.fit(aapd_X_train, aapd_y_train)
final_train_time_sec = time.perf_counter() - final_train_start

joblib.dump(final_model, f"{output_dir}/AAPD_SVC_final_model.joblib", compress=5)

print(f"Final best-configuration training time: {final_train_time_sec:.2f} seconds")

Final best-configuration training time: 497.02 seconds


In [ ]:
aapd_y_val_pred = final_model.predict(aapd_X_val)

val_miF1 = f1_score(aapd_y_val, aapd_y_val_pred, average="micro")
val_maF1 = f1_score(aapd_y_val, aapd_y_val_pred, average="macro")

print("Validation Micro-F1:", val_miF1)
print("Validation Macro-F1:", val_maF1)
#looks good

Validation Micro-F1: 0.7260834014717906
Validation Macro-F1: 0.5784803594865506


In [ ]:
start_inf = time.perf_counter()
aapd_y_test_pred = final_model.predict(aapd_X_test)
end_inf = time.perf_counter()

inference_time = end_inf - start_inf
print(f"Inference time: {inference_time:.3f} seconds")

test_miF1 = f1_score(aapd_y_test, aapd_y_test_pred, average="micro")
test_maF1 = f1_score(aapd_y_test, aapd_y_test_pred, average="macro")
print("Test Micro-F1:", test_miF1)
print("Test Macro-F1:", test_maF1)

Inference time: 0.562 seconds
Test Micro-F1: 0.6988636363636364
Test Macro-F1: 0.5554443666056249


In [ ]:
per_sample_ms = (inference_time / len(aapd_X_test)) * 1000  #in milliseconds
print(f"Per-sample inference: {per_sample_ms:.4f} ms")

Per-sample inference: 0.5622 ms


In [ ]:
print(classification_report(aapd_y_test, aapd_y_test_pred, target_names=mlb.classes_, zero_division=0))

                                                 precision    recall  f1-score   support

         Adaptation and Self-Organizing Systems       0.67      0.25      0.36         8
                                   Applications       0.00      0.00      0.00        14
                        Artificial Intelligence       0.49      0.50      0.49       111
                                  Combinatorics       0.56      0.63      0.59        60
                       Computation and Language       0.88      0.84      0.86        51
                       Computational Complexity       0.55      0.75      0.63        44
Computational Engineering, Finance, and Science       0.42      0.21      0.28        24
                         Computational Geometry       0.58      0.50      0.54        22
                      Computational Linguistics       0.80      0.80      0.80        15
               Computer Science and Game Theory       0.68      0.86      0.76        29
        Computer Vis

In [ ]:
report_dict = classification_report(
    aapd_y_test,
    aapd_y_test_pred,
    target_names=mlb.classes_,
    zero_division=0,
    output_dict=True)

report_df = pd.DataFrame(report_dict).T
report_df.to_csv(f"{output_dir}/AAPD_SVC_classification_report.csv")

In [ ]:
result = {
    "model": "LinearSVC_TFIDF",
    "dataset": "AAPD",
    "best_params": json.dumps(gs.best_params_),
    "cv_score": gs.best_score_,
    "gs_refit_time_sec": train_time, #gs+refit on train
    "final_train_time_sec": final_train_time_sec, #one fit of the selected config
    "val_f1_micro": val_miF1,
    "val_f1_macro": val_maF1,
    "test_f1_micro": test_miF1,
    "test_f1_macro": test_maF1,
    "inference_time_sec": inference_time, #test set
    "inference_per_sample_ms": per_sample_ms,}  #the other dataset will be of different size, for more fair comparison

df = pd.DataFrame([result])
df.to_csv(f"{output_dir}/AAPD_SVC_results.csv", index=False)